In [2]:
%matplotlib inline
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import json
import os
import datetime as dt
from tqdm import tqdm
import pandas as pd
import numpy as np

In [8]:
def f2cat(filename: str) -> str:
    return filename.split('.')[0]

class Simplified():
    def __init__(self, input_path='./train_simplified'):
        self.input_path = input_path

    def list_all_categories(self):
        files = os.listdir(os.path.join(self.input_path))
        return sorted([f2cat(f) for f in files], key=str.lower)

    def read_training_csv(self, category, nrows=None, usecols=None, drawing_transform=False):
        df = pd.read_csv(os.path.join(self.input_path, category + '.csv'),
                         nrows=nrows, parse_dates=['timestamp'], usecols=usecols)
        if drawing_transform:
            df['drawing'] = df['drawing'].apply(json.loads)
        return df

In [9]:
s = Simplified('./train_simplified')
NCSVS = 100
categories = s.list_all_categories()

In [11]:
for y, cat in tqdm(enumerate(categories)):
    df = s.read_training_csv(cat, nrows=10000)
    df['y'] = y
    df['cv'] = (df.key_id // 10 ** 7) % NCSVS
    for k in range(NCSVS):
        filename = 'train_k{}.csv'.format(k)
        chunk = df[df.cv == k]
        chunk = chunk.drop(['key_id'], axis=1)
        if y == 0:
            chunk.to_csv(filename, index=False)
        else:
            chunk.to_csv(filename, mode='a', header=False, index=False)
            
for k in tqdm(range(NCSVS)):
    filename = 'train_k{}.csv'.format(k)
    if os.path.exists(filename):
        df = pd.read_csv(filename)
        df['rnd'] = np.random.rand(len(df))
        df = df.sort_values(by='rnd').drop('rnd', axis=1)
        df.to_csv(filename + '.gz', compression='gzip', index=False)
        os.remove(filename)
print(df.shape)

340it [08:06,  1.43s/it]
100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [26:56<00:00, 16.17s/it]

(34065, 7)


In [25]:
import numpy as np
import six
import tensorflow as tf
import time
import os
import sklearn
%matplotlib inline
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import os
import ast
import datetime as dt
import seaborn as sns
import cv2
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import *
from keras.metrics import categorical_accuracy, top_k_categorical_accuracy, categorical_crossentropy
from keras.models import Sequential, Model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam
from keras.applications import MobileNet
from keras.applications.densenet import DenseNet121, DenseNet169
from keras.applications.mobilenet import preprocess_input
start = dt.datetime.now()

In [26]:
BASE_SIZE = 256 
NCSVS = 100 
NCATS = 340 
STEPS = 1000 
EPOCHS = 30 
size = 128 
batchsize = 256 
np.random.seed(seed=1987)
tf.compat.v1.set_random_seed(seed=1987) 

In [27]:
def apk(actual, predicted, k=3):
    
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    if not actual:
        return 0.0

    return score / min(len(actual), k)

def mapk(actual, predicted, k=3):
   
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

In [28]:
def top_3_accuracy(y_true, y_pred):
    return top_k_categorical_accuracy(y_true, y_pred, k=3)

In [29]:
input_layer = Input((size, size, 1))

common_layer = Reshape((size*size,), input_shape=(size, size, 1), name='common_layer')(input_layer)
common_layer = RepeatVector(3)(common_layer)
common_layer = Reshape((3, size, size), name='rs')(common_layer)
common_layer = Permute((3, 2, 1))(common_layer)

pretrained_model = DenseNet169(weights='imagenet', include_top=True, input_tensor=common_layer, classes=1000)
layer = pretrained_model.layers[-2].output

layer = Dense(NCATS, activation='softmax')(layer)

model = Model(input_layer, layer)

In [30]:
def draw_cv2(raw_strokes, size=256, lw=6, time_color=True):
    img = np.zeros((BASE_SIZE, BASE_SIZE), np.uint8)
    for t, stroke in enumerate(raw_strokes):
        for i in range(len(stroke[0]) - 1):
            color = 255 - min(t, 10) * 13 if time_color else 255
            _ = cv2.line(img, (stroke[0][i], stroke[1][i]),
                         (stroke[0][i + 1], stroke[1][i + 1]), color, lw)
    if size != BASE_SIZE:
        return cv2.resize(img, (size, size))
    else:
        return img

def image_generator_xd(size, batchsize, ks, lw=6, time_color=True):
    while True:
        for k in np.random.permutation(ks):
            filename = os.path.join('train_k{}.csv.gz'.format(k))
            for df in pd.read_csv(filename, chunksize=batchsize):
                df['drawing'] = df['drawing'].apply(ast.literal_eval)
                x = np.zeros((len(df), size, size, 1))
                for i, raw_strokes in enumerate(df.drawing.values):
                    x[i, :, :, 0] = draw_cv2(raw_strokes, size=size, lw=lw,
                                             time_color=time_color)
                x = preprocess_input(x).astype(np.float32)
                y = keras.utils.to_categorical(df.y, num_classes=NCATS)
                
                yield x, y

def df_to_image_array_xd(df, size, lw=6, time_color=True):
    df['drawing'] = df['drawing'].apply(ast.literal_eval)
    x = np.zeros((len(df), size, size, 1))
    for i, raw_strokes in enumerate(df.drawing.values):
        x[i, :, :, 0] = draw_cv2(raw_strokes, size=size, lw=lw, time_color=time_color)
    x = preprocess_input(x).astype(np.float32)
    return x

In [31]:
valid_df = pd.read_csv(os.path.join('train_k{}.csv.gz'.format(NCSVS - 1)), nrows=34000)
x_valid = df_to_image_array_xd(valid_df, size)
y_valid = keras.utils.to_categorical(valid_df.y, num_classes=NCATS)
print(x_valid.shape, y_valid.shape)
print('Validation array memory {:.2f} GB'.format(x_valid.nbytes / 1024.**3 ))

(34000, 128, 128, 1) (34000, 340)
Validation array memory 2.08 GB


In [32]:
train_datagen = image_generator_xd(size=size, batchsize=batchsize, ks=range(NCSVS - 1))

In [33]:
filepath = 'models/densenet169/accuracy_{val_categorical_accuracy:.10f}.hdf5'
checkpoint = ModelCheckpoint(filepath, monitor='val_categorical_accuracy', verbose=1, save_best_only=True, mode='max')

callbacks = [
    ReduceLROnPlateau(monitor='val_categorical_accuracy', factor=0.7, patience=10, mode='max', cooldown=3, verbose=1),
    checkpoint
]

model.fit_generator(
    train_datagen, steps_per_epoch=STEPS, epochs=EPOCHS, verbose=1,
    validation_data=(x_valid, y_valid),
    callbacks = callbacks
)

Instructions for updating:
Please use Model.fit, which supports generators.


RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.